In [ ]:
!pip install pennylane pennylane-lightning-gpu --upgrade
!pip install custatevec-cu12 cuquantum-cu12

In [ ]:
# =========================
# IMPORTS
# =========================
import torch
import torch.nn as nn
import pennylane as qml
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
import time
import math

# =========================
# CONFIG
# =========================
torch.manual_seed(42)

n_qubits = 6
n_layers = 3
input_dim = 4          # after periodic encoding
hidden_dim = 128
vocab_size = 10
batch_size = 32
epochs = 15

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [ ]:
# =========================
# DATASET (MODULO TASK)
# =========================
def generate_data(n=4000):
    a = torch.randint(0, 10, (n,))
    b = torch.randint(0, 10, (n,))

    x_raw = torch.stack([a, b], dim=1).float() / 9.0
    y = (a + b) % 10

    # ✅ PERIODIC ENCODING (CRITICAL FIX)
    x = torch.cat([
        torch.sin(math.pi * x_raw),
        torch.cos(math.pi * x_raw)
    ], dim=1)

    return x, y

x, y = generate_data(4000)

dataset = TensorDataset(x, y)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

In [ ]:
# =========================
# QUANTUM DEVICE
# =========================
dev = qml.device("lightning.gpu", wires=n_qubits)

# =========================
# QRL CIRCUIT
# =========================
@qml.qnode(dev, interface="torch", diff_method="adjoint")
def qrl_circuit(inputs, weights):

    qml.AngleEmbedding(inputs, wires=range(n_qubits))

    qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))

    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]


weight_shapes = {"weights": (n_layers, n_qubits, 3)}
qlayer = qml.qnn.TorchLayer(qrl_circuit, weight_shapes)

In [ ]:
# Torch Layer Wrapper
weight_shapes = {"weights": (n_layers, n_qubits, 3)}
qlayer = qml.qnn.TorchLayer(qrl_circuit, weight_shapes)

In [ ]:
# =========================
# CLASSICAL MODEL (STRONG)
# =========================
class ClassicalModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, vocab_size)
        )

    def forward(self, x):
        return self.net(x.to(device))

In [ ]:
# =========================
# HYBRID QUANTUM MODEL
# =========================
class HybridQuantumModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, n_qubits),
            nn.Tanh()
        )

        self.q_layer = qlayer

        self.decoder = nn.Sequential(
            nn.Linear(n_qubits, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, vocab_size)
        )

        self.classical_head = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, vocab_size)
        )

        self.alpha = nn.Parameter(torch.tensor(0.0))

    def forward(self, x):

        x = x.to(device)

        # Classical path
        classical_logits = self.classical_head(x)

        # Quantum path
        h = self.encoder(x)
        h = h * (2 * math.pi)   # ✅ FULL ROTATION RANGE

        q_out = torch.stack([self.q_layer(h_i) for h_i in h])

        quantum_logits = self.decoder(q_out)

        # Stable fusion
        alpha = torch.sigmoid(self.alpha)

        logits = alpha * quantum_logits + (1 - alpha) * classical_logits

        return logits

In [ ]:
# =========================
# TRAIN FUNCTION
# =========================
def train(model, dataloader, optimizer, loss_fn, epochs):

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0

        pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{epochs}")

        for xb, yb in pbar:

            xb, yb = xb.to(device), yb.to(device)

            optimizer.zero_grad()

            logits = model(xb)
            loss = loss_fn(logits, yb)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            epoch_loss += loss.item()
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})

        print(f"\nEpoch {epoch+1} Avg Loss: {epoch_loss/len(dataloader):.4f}")

In [ ]:
# =========================
# EVALUATE
# =========================
def evaluate(model, dataloader):

    model.eval()

    correct = 0
    total = 0

    start_time = time.time()

    with torch.no_grad():
        for xb, yb in dataloader:

            xb, yb = xb.to(device), yb.to(device)

            logits = model(xb)
            preds = logits.argmax(dim=1)

            correct += (preds == yb).sum().item()
            total += yb.size(0)

    end_time = time.time()

    accuracy = correct / total
    time_per_sample = (end_time - start_time) / total

    return accuracy, time_per_sample


In [ ]:
# =========================
# RUN
# =========================
loss_fn = nn.CrossEntropyLoss()

print("\n🧠 Training Classical Model...\n")
classical_model = ClassicalModel().to(device)
optimizer_c = torch.optim.Adam(classical_model.parameters(), lr=1e-3)

train(classical_model, dataloader, optimizer_c, loss_fn, epochs)
acc_c, t_c = evaluate(classical_model, dataloader)

print("\n⚛️ Training Hybrid Quantum Model...\n")
quantum_model = HybridQuantumModel().to(device)
optimizer_q = torch.optim.Adam(quantum_model.parameters(), lr=8e-4)

train(quantum_model, dataloader, optimizer_q, loss_fn, epochs)
acc_q, t_q = evaluate(quantum_model, dataloader)

print("\n📊 FINAL COMPARISON\n")
print(f"Classical Accuracy: {acc_c*100:.2f}%")
print(f"Quantum Accuracy:   {acc_q*100:.2f}%\n")

print(f"Classical Time/sample: {t_c:.6f} sec")
print(f"Quantum Time/sample:   {t_q:.6f} sec")

print("\nAlpha (quantum weight):", torch.sigmoid(quantum_model.alpha).item())


🧠 Training Classical Model...



Epoch 1/15: 100%|██████████| 125/125 [00:00<00:00, 179.13it/s, loss=1.5230]



Epoch 1 Avg Loss: 2.0021


Epoch 2/15: 100%|██████████| 125/125 [00:01<00:00, 112.30it/s, loss=0.7895]



Epoch 2 Avg Loss: 1.1315


...


Epoch 13 Avg Loss: 0.0087


Epoch 14/15: 100%|██████████| 125/125 [07:41<00:00,  3.69s/it, loss=0.0048]



Epoch 14 Avg Loss: 0.0058


Epoch 15/15: 100%|██████████| 125/125 [07:38<00:00,  3.67s/it, loss=0.0046]



Epoch 15 Avg Loss: 0.0042

📊 FINAL COMPARISON

Classical Accuracy: 100.00%
Quantum Accuracy:   100.00%

Classical Time/sample: 0.000021 sec
Quantum Time/sample:   0.040005 sec

Alpha (quantum weight): 0.69431072473526


In [ ]:
# =========================
# RESULTS
# =========================
print("\n📊 FINAL COMPARISON\n")

print(f"Classical Accuracy: {acc_c*100:.2f}%")
print(f"Quantum Accuracy:   {acc_q*100:.2f}%\n")

print(f"Classical Time/sample: {t_c:.6f} sec")
print(f"Quantum Time/sample:   {t_q:.6f} sec")


📊 FINAL COMPARISON

Classical Accuracy: 100.00%
Quantum Accuracy:   100.00%

Classical Time/sample: 0.000021 sec
Quantum Time/sample:   0.040005 sec
